## ▶ What you'll see when you run this
- A **matplotlib heatmap** of an attention matrix showing which words each word 'looks at' — the engine inside every Transformer, made visible.

**Time:** ~20 min · **Cost:** free (pure NumPy + matplotlib, no model calls) · **Keys:** none

# Week 8 · Notebook 4 — Attention **From Scratch** (NumPy)
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

The Transformer that powers every modern LLM is built on one operation: **scaled dot-product attention**. We compute it by hand in NumPy on a tiny toy sentence, then **visualize** which words attend to which.

The recipe:
1. Turn each word into **Q** (query), **K** (key), **V** (value) vectors.
2. **scores = Q · Kᵀ / √d** — how well each query matches each key.
3. **softmax** each row → attention weights that sum to 1.
4. **weighted sum of V** → each word's new, context-aware vector.

> Pure NumPy. No keys, no GPU, no full Transformer — just the core engine.

## 0. Wow in 5 seconds — our toy sentence as numbers
Each word gets a small random **embedding** (a row of numbers). Real models learn these; we just make them up so the math is visible.

In [ ]:
import numpy as np
np.random.seed(0)

words = ['the', 'cat', 'sat', 'on', 'mat']
n = len(words)          # sequence length
d = 4                   # embedding dimension (tiny on purpose)
X = np.random.randn(n, d)
print('sentence:', words)
print('embeddings X shape:', X.shape, '(one row per word)')
print(np.round(X, 2))

## 1. Build Q, K, V with small random weight matrices
Attention first projects each embedding into a **query**, a **key**, and a **value** using learned weight matrices `Wq, Wk, Wv`. We use random ones.

In [ ]:
Wq = np.random.randn(d, d) * 0.5
Wk = np.random.randn(d, d) * 0.5
Wv = np.random.randn(d, d) * 0.5

Q = X @ Wq    # queries
K = X @ Wk    # keys
V = X @ Wv    # values
print('Q, K, V shapes:', Q.shape, K.shape, V.shape)
assert Q.shape == (n, d) and K.shape == (n, d) and V.shape == (n, d)
print('assert passed: Q/K/V are (n_words, d) ✔')

## 2. Fill-in #1 — scaled dot-product scores
**Your turn.** Compute `scores = Q · Kᵀ / √d`. Dividing by **√d** keeps the numbers from blowing up as the dimension grows (the 'scaled' part).

In [ ]:
def attention_scores(Q, K):
    d = Q.shape[1]
    # TODO: matrix-multiply Q by K transpose, then divide by sqrt(d)
    return (Q @ K.T) / np.sqrt(d)

scores = attention_scores(Q, K)
print('scores shape:', scores.shape, '(word-by-word match strength)')
assert scores.shape == (n, n)
print('assert passed: scores is n x n ✔')

## 3. Fill-in #2 — softmax each row into weights
**Your turn.** Softmax turns each row of scores into **probabilities that sum to 1** — the attention weights. We subtract the row max first for numerical stability.

In [ ]:
def softmax_rows(M):
    M = M - M.max(axis=1, keepdims=True)   # stability
    e = np.exp(M)
    # TODO: divide each row by its sum so the row totals 1
    return e / e.sum(axis=1, keepdims=True)

weights = softmax_rows(scores)
print('each row sums to 1?')
print(np.round(weights.sum(axis=1), 6))
assert np.allclose(weights.sum(axis=1), 1.0)
assert (weights >= 0).all()
print('assert passed: weights are a valid probability distribution ✔')

## 4. Fill-in #3 — weighted sum of V (the output)
**Your turn.** Each word's new vector is the **attention-weighted average of all value vectors**: `output = weights · V`. This is where context actually flows between words.

In [ ]:
def attention(Q, K, V):
    scores = attention_scores(Q, K)
    weights = softmax_rows(scores)
    # TODO: multiply the weights by V to mix the value vectors
    out = weights @ V
    return out, weights

output, weights = attention(Q, K, V)
print('output shape:', output.shape, '(one context-aware vector per word)')
assert output.shape == (n, d)
print('assert passed: attention output is (n_words, d) ✔')

## 5. Visualize the attention matrix as a heatmap
Row *i* shows how much word *i* attends to every other word. Brighter = more attention. **This is the picture researchers stare at to interpret models.**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights, cmap='viridis')
ax.set_xticks(range(n)); ax.set_xticklabels(words)
ax.set_yticks(range(n)); ax.set_yticklabels(words)
ax.set_xlabel('attends to (key)')
ax.set_ylabel('query word')
ax.set_title('Scaled Dot-Product Attention')
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{weights[i, j]:.2f}', ha='center', va='center',
                color='white', fontsize=8)
fig.colorbar(im, ax=ax, label='attention weight')
plt.tight_layout()
plt.show()
print('Each ROW sums to 1 — a word distributes its attention across the sentence.')

## 6. Multi-head attention — the concept
Real Transformers run attention **several times in parallel** (multiple 'heads'), each with its **own** Wq/Wk/Wv. One head might track grammar, another long-range references. The heads' outputs are concatenated. We don't build a full Transformer here — just see that more heads = more *kinds* of relationships captured at once.

In [ ]:
def make_head(seed):
    rng = np.random.RandomState(seed)
    Wq, Wk, Wv = (rng.randn(d, d) * 0.5 for _ in range(3))
    out, w = attention(X @ Wq, X @ Wk, X @ Wv)
    return w

heads = [make_head(s) for s in (1, 2, 3)]
print('built', len(heads), 'attention heads, each (n x n):',
      [h.shape for h in heads])
# different heads -> different attention patterns
assert not np.allclose(heads[0], heads[1])
print('assert passed: heads learn DIFFERENT patterns ✔')

## 7. Try it on YOUR words (S-exercise)
Swap in your own short sentence below and re-run to get a fresh heatmap. Notice the weights still form a valid distribution (each row sums to 1).

In [ ]:
my_words = ['ai', 'is', 'fun', 'today']   # <- change me
m = len(my_words)
Xv = np.random.randn(m, d)
Qv, Kv, Vv = Xv @ Wq, Xv @ Wk, Xv @ Wv
out_v, w_v = attention(Qv, Kv, Vv)
print('your sentence:', my_words)
print('attention rows sum to 1:', np.round(w_v.sum(axis=1), 4))
assert np.allclose(w_v.sum(axis=1), 1.0)
print('assert passed: your attention is a valid distribution ✔')

---
### Recap
Q/K/V → **QKᵀ/√d** → **softmax** → **weighted sum of V**. That four-step engine, stacked in many layers and many heads, **is** the Transformer behind every LLM. You computed it in NumPy and saw the attention heatmap that researchers use to interpret what a model is doing.